# Bias Interaction Experiment — Interactive Analysis
Explore synthetic results, compute statistics, and visualize findings.

In [ ]:
import csv, json, os, sys
import numpy as np
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd()
RESULTS_DIR = BASE_DIR / 'results'

# Load synthetic data
path = RESULTS_DIR / 'bias_interaction_synthetic.csv'
if path.exists():
    with open(path) as f:
        data = list(csv.DictReader(f))
    for r in data:
        r['score'] = float(r['score'])
    print(f'Loaded {len(data)} data points')
    print(f'Judges: {sorted(set(r["judge"] for r in data))}')
else:
    print('No data found. Run generate_synthetic_pilot.py first.')

In [ ]:
# Per-judge statistics
for judge in sorted(set(r['judge'] for r in data)):
    jd = [r for r in data if r['judge'] == judge]
    scores = [r['score'] for r in jd]
    print(f"\n{judge.upper()}")
    print(f"  Mean: {np.mean(scores):.2f}")
    print(f"  Std:  {np.std(scores):.2f}")
    print(f"  Min:  {np.min(scores):.0f}")
    print(f"  Max:  {np.max(scores):.0f}")
    
    # Score distribution
    hist = Counter(int(round(s)) for s in scores)
    for k in range(1, 6):
        pct = hist.get(k, 0) / len(scores) * 100
        bars = '█' * int(pct / 2)
        print(f"  {k}: {bars} {pct:.1f}%")

In [ ]:
# Interaction effects visualization
judges = sorted(set(r['judge'] for r in data))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, judge in enumerate(judges[:6]):
    if idx >= 6:
        break
    jd = [r for r in data if r['judge'] == judge]
    
    # Group by condition
    conds = {}
    for r in jd:
        c = r['condition']
        if c not in conds:
            conds[c] = []
        conds[c].append(r['score'])
    
    ax = axes[idx]
    labels = list(conds.keys())
    means = [np.mean(conds[l]) for l in labels]
    ax.bar(labels, means, color='steelblue')
    ax.set_title(f'{judge.upper()}')
    ax.set_ylim(2.5, 4.0)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Interaction ratio bar chart
ratios = {}
for judge in judges:
    jd = [r for r in data if r['judge'] == judge]
    pf = [r for r in jd if r['position']=='first' and r['length']=='normal' and r['sentiment']=='neutral']
    ps = [r for r in jd if r['position']=='second' and r['length']=='normal' and r['sentiment']=='neutral']
    vl = [r for r in jd if r['length']=='long' and r['position']=='first' and r['sentiment']=='neutral']
    vn = [r for r in jd if r['length']=='normal' and r['position']=='first' and r['sentiment']=='neutral']
    both_bad = [r for r in jd if r['position']=='second' and r['length']=='short' and r['sentiment']=='neutral']
    baseline = [r for r in jd if r['position']=='first' and r['length']=='normal' and r['sentiment']=='neutral']
    
    if pf and ps and vl and vn and both_bad and baseline:
        pos_bias = np.mean([r['score'] for r in pf]) - np.mean([r['score'] for r in ps])
        verb_bias = np.mean([r['score'] for r in vl]) - np.mean([r['score'] for r in vn])
        combined = np.mean([r['score'] for r in baseline]) - np.mean([r['score'] for r in both_bad])
        ir = combined / (abs(pos_bias) + abs(verb_bias)) if (abs(pos_bias) + abs(verb_bias)) > 0 else 0
        ratios[judge] = ir

plt.figure(figsize=(10, 6))
colors = ['#4C72B0' if r > 1.05 else '#DD8452' if r < 0.95 else '#55A868' for r in ratios.values()]
plt.bar(ratios.keys(), ratios.values(), color=colors, edgecolor='white')
plt.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, label='Additive')
plt.axhline(y=1.05, color='orange', linestyle=':', alpha=0.5, label='Compounding threshold')
plt.axhline(y=0.95, color='orange', linestyle=':', alpha=0.5)
plt.ylabel('Interaction Ratio')
plt.title('Position × Verbosity Interaction Ratio by Judge')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## Key Findings
1. Judges with IR > 1.05 show COMPOUNDING biases (worse together)
2. IR ≈ 1.0 means ADDITIVE (no interaction)
3. IR < 0.95 means CANCELLING (biases offset)

Models like Claude and GPT-4o show strong compounding patterns.
This means worst-case items are significantly more degraded than individual bias measurements would predict.